# 3 · Do use-case profiles need contextual adaptation?

**Research question (RQ 3):** *The current IQB aggregates performance across six use cases with uniform weighting. This question investigates whether context-specific use-case profiles, such as a "developing country" profile that excludes or de-prioritizes use cases like gaming or video streaming, produce more meaningful and actionable scores. Equal weighting may result in countries being penalized for their performance in online gaming.*

We compare five profiles. Each changes only the use-case weights; requirement weights and thresholds stay the same:

| Profile | Web browsing | Video streaming | Audio streaming | Video conferencing | Online backup | Gaming | Rationale |
|---|---|---|---|---|---|---|---|
| **Uniform** (default) | 1 | 1 | 1 | 1 | 1 | 1 | current IQB |
| **No gaming** | 1 | 1 | 1 | 1 | 1 | 0 | the proposal's example |
| **Essential services** | 1 | 1 | 1 | 1 | 0 | 0 | information, media, communication; drops backup and gaming |
| **Basic access** | 1 | 0 | 0 | 1 | 0 | 0 | web access and video calls only (the proposal's "primary connectivity needs") |
| **IQB Edu** | 1 | 1 | 0.5 | 1 | 0.5 | 0 | |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import iqb_tools as T

T.set_style()
rng = np.random.default_rng(6)  # random use-case weights (section on weights vs percentile)
cfg = T.config_table()

# Sample: at least the minimum number of tests in 2025 (1a) and a World Bank income group
ctx = T.read_table("country_context").set_index("country_code")  # written by 0-setup
d = ctx[(T.binding_tests(ctx) >= T.MIN_TESTS) & ctx.income_group.notna()].copy()
d["income"] = pd.Categorical(d.income_group, T.INCOME_ORDER, ordered=True)

# 2025 pass rates per country: the test-weighted average of the 12 months
df25 = T.load_level("country", months=pd.date_range("2025-01-01", "2025-12-01", freq="MS"))
df25 = df25[df25.country_code.isin(d.index)]
tests = T.distinct_tests(cfg)
C = T.pool_pass_rates(df25, T.pass_rates(df25, tests), "country_code", tests).loc[d.index]

PROFILES = {  # use-case weights, as in the table above
    "Uniform": {u: 1 for u in T.USE_CASES},
    "No gaming": {u: 1 for u in T.USE_CASES if u != "gaming"},
    "Essential services": {"web browsing": 1, "video streaming": 1, "audio streaming": 1, "video conferencing": 1},
    "Basic access": {"web browsing": 1, "video conferencing": 1},
    "IQB Edu": {"web browsing": 1, "video streaming": 1, "audio streaming": 0.5,
                "video conferencing": 1, "online backup": 0.5, "gaming": 0},
}
tables = {name: T.with_use_case_weights(cfg, w) for name, w in PROFILES.items()}
scores = {name: T.iqb_scores(C, tb, T.Q_GRID) for name, tb in tables.items()}  # countries x P1..P99
LEVELS = [25, 50, 75, 95]
print(len(d), "countries:", d.income.value_counts(sort=False).to_dict())

## Which use cases hold countries back? 

Use cases share thresholds, so every profile is a different weighting of the same ten tests.

In [ ]:
ew = pd.DataFrame({name: T.effective_test_weights(tb) for name, tb in tables.items()}).fillna(0)
ew.loc[ew.Uniform.sort_values(ascending=False).index].round(3)

Each column is a profile, and each value is one test's weight in the IQB score under that profile: its share of the score when it passes. Each column adds up to 1. 

The rows are the ten tests, sorted by their weight under *Uniform*; *loss<0.01* is packet loss below 1%, *latency<100* is latency below 100 ms, *download>10* is download above 10 Mbit/s, and so on. 

A test's weight adds up its weights in every use case; so the tests shared by most use cases (1% loss, 100 ms latency) carry the most weight. A weight of 0 means that no use case in the profile uses the test: the 10 ms latency test belongs only to gaming, so it drops out of every profile without gaming, and *Basic access* also drops the 5 Mbit/s upload test.

The table shows that dropping gaming (*No gaming*, *IQB Edu*) removes only one test the other use cases do not have: the 10 ms latency test, with a weight of 0.05. 

The rest of gaming's weight goes back to tests that other use cases already contain. 

Under *No gaming*, the 1% loss and 100 ms latency tests rise from 0.20 and 0.16 to 0.24 and 0.20. 

*IQB Edu* also halves online backup and audio streaming, which moves weight away from the 25 Mbit/s upload test (0.13 → 0.10) and toward the same loss and latency tests (0.23 and 0.20).

In [ ]:
uc = T.use_case_scores(C, cfg, LEVELS)
uc_tbl = pd.concat({f"P{q}": pd.DataFrame({u: uc[u][q].groupby(d.income, observed=True).mean() for u in T.USE_CASES})
                    for q in LEVELS})
uc_tbl.index = [f"{T.INCOME_SHORT[i]} · {p}" for p, i in uc_tbl.index]

fig, ax = plt.subplots(figsize=(8.5, 5.2))
cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list("seq", [T.BLUE_RAMP[100], T.BLUE_RAMP[700]])
im = ax.imshow(uc_tbl.to_numpy(), cmap=cmap, vmin=0, vmax=1, aspect="auto")
for (i, j), v in np.ndenumerate(uc_tbl.to_numpy()):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9, color="white" if v > 0.55 else T.INK)
for k in range(4, len(uc_tbl), 4):
    ax.axhline(k - 0.5, color=T.SURFACE, lw=3)
ax.set_xticks(range(6), [u.replace(" ", "\n") for u in T.USE_CASES])
ax.set_yticks(range(len(uc_tbl)), uc_tbl.index)
ax.grid(False)
ax.set_title("Mean use-case score by income group and percentile")
fig.colorbar(im, ax=ax, fraction=0.035)
T.savefig(fig, "3_use_case_scores_by_income")

Each cell is the average score of one use case (column) for the countries of one income group at one percentile (row). 

A use-case score is the weighted share of its requirements that pass at that percentile, from 0 (none pass) to 1 (all pass). The rows come in four blocks, one per percentile from P25 at the top to P95 at the bottom, with the income groups from low to high inside each block. Darker cells are higher scores, and every value is printed in its cell.

Scores rise from top to bottom within each block, as income rises, and from one block to the next, as the percentile rises. In every row, video conferencing and gaming are the two lightest columns, and at P25 almost the whole block is light: even high-income countries average only 0.07 on gaming and 0.23 on video conferencing.

The heatmap shows that gaming is the lowest use case in every income group and at every percentile (all 16 rows), and video conferencing (50 ms, 0.5% loss, 25 Mbit/s upload) is the second lowest in every row. At P25 and P50, low-income countries score low on *every* use case, including web browsing (0.04 and 0.17). Their low IQB is therefore not mainly a gaming effect.

### Is gaming a penalty, and for whom?

We define the gaming penalty as IQB(No gaming) − IQB(Uniform), across the full percentile range. We also recompute it with the gaming thresholds from the framework table (minimum: 10/10 Mbit/s, 100 ms, 1%) instead of the library's (25/25 Mbit/s, 10 ms, 0.5%).


In [ ]:
fw_gaming = cfg.copy()
sel = fw_gaming.use_case == "gaming"
fw_gaming.loc[sel, "threshold"] = [T.FRAMEWORK_MIN["gaming"][m] for m in fw_gaming.loc[sel, "metric"]]
df25 = T.load_level("country", months=pd.date_range("2025-01-01", "2025-12-01", freq="MS"))
df25 = df25[df25.country_code.isin(d.index)]
tests_fw = T.distinct_tests(fw_gaming)
C_fw = T.pool_pass_rates(df25, T.pass_rates(df25, tests_fw), "country_code", tests_fw).loc[d.index]
S_fw = T.iqb_scores(C_fw, fw_gaming, T.Q_GRID)

penalty_lib = scores["No gaming"] - scores["Uniform"]
penalty_fw = T.iqb_scores(C_fw, T.with_use_case_weights(fw_gaming, PROFILES["No gaming"]), T.Q_GRID) - S_fw

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, pen, title in [(axes[0], penalty_lib, "library gaming thresholds (25/25 Mbit/s, 10 ms, 0.5%)"),
                       (axes[1], penalty_fw, "framework minimum (10/10 Mbit/s, 100 ms, 1%)")]:
    for inc in T.INCOME_ORDER:
        ax.plot(T.Q_GRID, pen[d.income == inc].mean(), color=T.INCOME_COLORS[inc], label=T.INCOME_SHORT[inc])
    ax.axhline(0, color=T.AXIS, lw=1)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("percentile"); ax.set_xlim(0, 100)
axes[0].set_ylabel("mean score gain from dropping gaming")
axes[1].legend(*axes[0].get_legend_handles_labels(), title="income", loc="upper right", ncol=2)
fig.suptitle("How much does the gaming use case lower IQB?", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
T.savefig(fig, "3_gaming_penalty")

pd.DataFrame({f"library P{q}": penalty_lib[q].groupby(d.income, observed=True).mean() for q in LEVELS} |
             {f"framework P{q}": penalty_fw[q].groupby(d.income, observed=True).mean() for q in LEVELS}).round(3)

The table gives the average gaming penalty, the score gain from dropping gaming (IQB under *No gaming* minus IQB under *Uniform*), for each income group (rows) at P25, P50, P75 and P95. The first four columns use the library's gaming thresholds and the last four the framework's minimum thresholds. A positive value means that gaming lowers the score: 0.055 means that the high-income countries' average P25 score rises by 0.055 without gaming.

The figure plots the same gain at every percentile. The x-axis is the percentile from P1 to P99, the y-axis is the mean gain from dropping gaming, and each line is an income group (darker for higher income); the horizontal line marks no change. With the library's thresholds (left), every curve is above zero: it starts at zero at the lowest percentiles, peaks at 0.05–0.06 and then falls. The peak comes earlier the higher the income: at about P34 for high-income countries, P50 for upper-middle-income, P65 for lower-middle-income and P91 for low-income countries. With the framework's thresholds (right), every curve stays at or slightly below zero, never lower than about −0.02, so dropping gaming then lowers scores a little.

Gaming lowers scores by at most about 0.05 on average, and who it penalizes depends on the percentile.

With the library's thresholds, the penalty at P25 grows with income (0.005 for low-income countries, 0.055 for high-income ones), and at P50 it is larger for middle- and high-income countries (about 0.05) than for low-income ones (0.02). At these percentiles low-income countries already fail the tests gaming shares with the other use cases, so removing gaming changes little for them. At P75–P95 the penalty is larger for low- and lower-middle-income countries, because the 10 ms latency test is the one they still fail at those percentiles.

With the framework table's gaming thresholds, gaming has the same requirements as web browsing and the penalty disappears: dropping gaming then changes scores by less than 0.02, and slightly downward. The "gaming penalty" is therefore mostly a consequence of the library's 10 ms threshold, not of including gaming at all. Aligning the library with the framework table (or documenting why gaming needs 10 ms) matters more than re-weighting.

## Do the alternative profiles produce more meaningful scores?

In [ ]:
def profile_metrics(name, q):
    s, base = scores[name][q], scores["Uniform"][q]
    diff = s - base
    return {
        "profile": name,
        "percentile": f"P{q}",
        **{f"Δ {T.INCOME_SHORT[i]}": diff[d.income == i].mean() for i in T.INCOME_ORDER},
        "Spearman with Uniform": spearmanr(s, base).statistic,
        "SD across countries": s.std(),
        "share at 1.0": (s > 0.999).mean(),
        "share at 0": (s < 0.001).mean(),
        "distinct values": s.round(6).nunique(),
        "ρ Internet users": spearmanr(s, d.internet_users_pct, nan_policy="omit").statistic,
        "ρ log GDP pc": spearmanr(s, d.log10_gdp_pc, nan_policy="omit").statistic,
        "median swing P50→P95": (scores[name][95] - scores[name][50]).median(),
    }


pm = pd.DataFrame([profile_metrics(n, q) for q in LEVELS for n in PROFILES]).set_index(["percentile", "profile"])
pm.round(3)

Each row is one profile at one percentile, compared with *Uniform* at the same percentile:

| Column | Meaning |
|---|---|
| *Δ Low* … *Δ High* | The average change in score for the income group, against *Uniform*. |
| *Spearman with Uniform* | The rank correlation between the profile's ranking of countries and the *Uniform* ranking (1 means the same order). |
| *SD across countries* | The spread of scores across countries: a larger spread separates countries better. |
| *share at 1.0*, *share at 0* | The share of countries with a perfect score or a score of zero, where differences between countries disappear. |
| *distinct values* | How many different scores the 201 countries receive. |
| *ρ Internet users*, *ρ log GDP pc* | The rank correlation of the score with the share of the population using the Internet and with GDP per capita, a check on validity. |
| *median swing P50→P95* | The median difference between a country's P95 and P50 scores under the profile. It does not depend on the row's percentile, so it repeats in every block. |

A profile would give more meaningful scores if it raised the spread and the number of distinct values without pushing more countries to 1.0, and if it raised the correlations with Internet use and GDP. For example, at P95, *No gaming* raises the low-income average by 0.048 and keeps a rank correlation of 0.897 with *Uniform*, but it puts 62.7% of countries at 1.0, against 35.3% under *Uniform*.

In [ ]:
# Colors: green, amber and pink are below 3:1 contrast on the background; every value is in the table above
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
names = list(PROFILES)
colors = dict(zip(names, T.SERIES[:5]))
x = np.arange(len(LEVELS))
slot, bar = 0.15, 0.14
for k, n in enumerate(names):
    sub = pm.xs(n, level="profile").loc[[f"P{q}" for q in LEVELS]]
    off = (k - (len(names) - 1) / 2) * slot
    axes[0].bar(x + off, sub["share at 1.0"], bar, color=colors[n], label=n)
    axes[1].bar(x + off, sub["SD across countries"], bar, color=colors[n])
    axes[2].bar(x + off, sub["ρ Internet users"], bar, color=colors[n])
for ax, t in zip(axes, ["share of countries at the ceiling (1.0)", "spread across countries (SD)",
                        "Spearman ρ with Internet users (%)"]):
    ax.set_xticks(x, [f"P{q}" for q in LEVELS])
    ax.set_title(t, fontsize=10)
    ax.grid(axis="x", visible=False)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
fig.suptitle("Profiles compared: discrimination and validity", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(rect=(0, 0.08, 1, 1))
fig.legend(*axes[0].get_legend_handles_labels(), ncol=5, fontsize=9, loc="lower center", bbox_to_anchor=(0.5, 0.0))
T.savefig(fig, "3_profiles_compared")

The figure plots three columns of the table. In each panel the x-axis is the percentile, with one bar per profile in the colors of the legend. The left panel's y-axis is the share of countries at the ceiling of 1.0, the middle one's is the spread of scores across countries (SD), and the right one's is the rank correlation with the share of the population using the Internet. Taller bars are worse in the left panel and better in the other two.

In the left panel, the four alternative profiles have identical bars, well above *Uniform*, at every percentile from P50 on, and the gap grows with the percentile (12% against 3% at P50, 63% against 35% at P95). In the middle panel, the bars are similar within each percentile, and the spread is largest at P50 for every profile. In the right panel, *Uniform* has the tallest bar, or close to it, at every percentile, and all bars drop at P95.

The table and figure show that the alternative profiles, including IQB Edu, do not produce more discriminating or more valid scores.

Discrimination gets worse where it is already weakest. At P95, every alternative profile pushes far more countries to the ceiling of 1.0 (63% instead of 35%), and at P75 42% instead of 16%, because the demanding tests that kept them below 1 lose weight. At P25 and P50 the spread rises slightly (SD 0.19 → 0.20–0.22 at P25, 0.32 → 0.34 at P50), but the number of distinct scores falls (at P50, from 60 to between 28 and 53).

Validity does not improve either. The rank correlation with Internet use and GDP per capita is essentially unchanged at P25 and P50, and slightly lower at P75 and P95 (with Internet use at P95: 0.52 under *Uniform*, 0.48–0.50 under the others). Rankings barely move at P25 and P50 (Spearman ρ ≥ 0.99 with the uniform profile), and only moderately at P95 (about 0.89), where the change is mostly a mass of new ties at 1.0.

Scores change little: between −0.01 and +0.07 on average by income group. For *No gaming*, *Essential services* and *IQB Edu* at P95, the rise is largest for low-income countries. So a "developing-country profile" makes developing countries look modestly better at high percentiles, without separating them from each other any better.

*IQB Edu* behaves like *No gaming*: the same ceiling shares, the same rankings and the same correlations with Internet use, with slightly smaller score gains (at P50, +0.02 to +0.05 by income group). Halving online backup and audio streaming changes little, because those use cases share their tests with the others.

## Which matters more, the weights or the percentile?

Following the OECD/JRC handbook on composite indicators, we draw 1,000 random use-case weight vectors uniformly from the simplex (Dirichlet(1, …, 1)), combine each with the percentiles P25, P50, P75 and P95, and split the variance of each country's score into the part due to the percentile and the part due to the weights.

In [ ]:
W_draws = rng.dirichlet(np.ones(6), size=1000)  # 1,000 x 6 use-case weight vectors
ucs = T.use_case_scores(C, cfg, LEVELS)
U = np.stack([ucs[u].to_numpy() for u in T.USE_CASES], axis=-1)  # countries x percentiles x use cases
grid = np.einsum("cqu,wu->cqw", U, W_draws)  # countries x percentiles x weight draws

total = grid.reshape(len(C), -1).var(axis=1)
v_pct = grid.mean(axis=2).var(axis=1)  # main effect of the percentile
v_w = grid.mean(axis=1).var(axis=1)  # main effect of the weights
share = pd.DataFrame(
    {"percentile": v_pct / total, "weights": v_w / total, "interaction": 1 - (v_pct + v_w) / total},
    index=C.index,
).fillna(0)
share["income"] = d.income
print("Median share of each country's score variance:")
share.groupby("income", observed=True).median().round(2)

For each country, the scores from the 1,000 random weightings at the four percentiles form 4,000 values. The table splits their variance into the part explained by the percentile alone, the part explained by the weights alone, and the rest (*interaction*: the effect of the weights depends on the percentile), and gives the median of each share over the countries of each income group. Each row adds up to about 1. For example, 0.97 means that for the median low-income country, 97% of the variation in its score comes from which percentile is used.

In [ ]:
spread_w = pd.DataFrame(
    np.percentile(grid, 95, axis=2) - np.percentile(grid, 5, axis=2), index=C.index, columns=LEVELS
)
print("Median 90% range of a country's score across random weights, by percentile:")
print(spread_w.groupby(d.income, observed=True).median().round(2).to_string())
swing = (scores["Uniform"][95] - scores["Uniform"][50]).groupby(d.income, observed=True).median().round(2)
print(f"\nfor comparison, median swing P50 → P95 under uniform weights: {swing.to_dict()}")

The first table shows how far the weights alone can move a score. For each country and percentile, it takes the range covering the middle 90% of the scores from the 1,000 random weightings, and gives the median of that range by income group (rows) and percentile (columns). A value of 0 means the weights make almost no difference, typically because the country passes no test (low-income countries at P25) or every test (high-income countries at P95) at that percentile; the largest value anywhere is 0.24. The last line gives, for comparison, how much the median country's score changes between P50 and P95 under uniform weights: from 0.18 for high-income countries to 0.64 for low-income ones.

The percentile choice matters far more than the use-case weights. For the median country in every income group, the percentile explains 96–97% of the variance of its score across P25–P95 and arbitrary weights, while the weights explain 1–2%. Even fully random weights move the median country's score within a 90% range of at most 0.24 at any percentile, against a median swing of 0.18–0.64 between P50 and P95.

The share depends on how wide a range of percentiles is compared. Over P50–P95 only, the percentile still dominates, but for high-income countries the weights then explain 12–16%, because their score at P50 depends on which demanding use cases are counted.

## Can scores under different profiles be compared?

The proposal's idea is to apply a developing-country profile *to developing countries*. If low- and lower-middle-income countries are scored with *Essential services* and the rest with *Uniform*, how much of the resulting comparison is an artifact of the profile switch?

In [ ]:
mixed = scores["Uniform"].copy()
dev = d.income.isin(["Low income", "Lower middle income"])
mixed.loc[dev] = scores["Essential services"].loc[dev]
rows = []
for q in LEVELS:
    r_u = scores["Uniform"][q].rank(ascending=False)
    r_m = mixed[q].rank(ascending=False)
    overtaken = sum(
        (mixed.loc[i, q] > mixed.loc[j, q]) and (scores["Uniform"].loc[i, q] <= scores["Uniform"].loc[j, q])
        for i in d.index[dev]
        for j in d.index[~dev]
    )
    rows.append(
        {
            "percentile": f"P{q}",
            "developing countries gaining rank": (r_m[dev] < r_u[dev]).sum(),
            "median rank gain (places)": (r_u[dev] - r_m[dev]).median(),
            "max rank gain (places)": (r_u[dev] - r_m[dev]).max(),
            "other countries overtaken (pairs)": int(overtaken),
        }
    )
pd.DataFrame(rows).set_index("percentile")

The table compares two global rankings at each percentile (rows): one with every country scored under *Uniform*, and a mixed one in which the 65 low- and lower-middle-income countries are scored under *Essential services* and the rest under *Uniform*:

| Column | Meaning |
|---|---|
| *developing countries gaining rank* | How many of the 65 rank higher in the mixed ranking. |
| *median rank gain (places)*, *max rank gain (places)* | The median and largest number of places they gain. Half places come from ties. |
| *other countries overtaken (pairs)* | The number of pairs in which a low- or lower-middle-income country is above an upper-middle- or high-income country in the mixed ranking but was not above it under *Uniform*. |

The number of countries gaining rank and the size of their gains grow with the percentile: at P25, 16 countries gain rank, by a median of 0 places; at P95, 49 gain, by a median of 6 places and up to 55. The number of pairs overtaken dips at P50 (250, against 314 at P25) and then rises to 1,179 at P95.

Scored on *Essential services*, up to 49 of the 65 low- and lower-middle-income countries gain rank at P95 (a median of 6 places, up to 55), overtaking other countries in 1,179 pairs, only because they are scored on an easier basket. A context-specific profile therefore makes scores **incomparable across contexts**. Profiles are only defensible for *within-context* uses (prioritizing among regions of one country, or among countries of one income group). They must not be used in global comparisons, and scores computed under different profiles must never be placed on the same map.

## Save

In [ ]:
out = pd.DataFrame({f"{n} P{q}": scores[n][q] for n in PROFILES for q in LEVELS})
out = d[["country", "income_group"]].join(out.round(4))
out.to_csv(T.TAB / "profile_scores.csv")
pm.to_csv(T.TAB / "profile_metrics.csv")
share.to_csv(T.TAB / "profile_variance_shares.csv")
print("saved profile_scores.csv", out.shape, ", profile_metrics.csv and profile_variance_shares.csv to", T.TAB)

## Summary

Use cases share the same ten tests, so re-weighting them has little leverage: average scores move by −0.01 to +0.07, rankings at P25 and P50 barely change, and random weights move a country's score far less than the choice of percentile.

The alternative profiles are not more discriminating. Dropping or halving demanding use cases sends 63% of countries to 1.0 at P95 (35% under *Uniform*) and leaves the correlation with Internet use and GDP unchanged or slightly lower.

The "gaming penalty" is a threshold issue. It comes from the library's 10 ms latency requirement, which is stricter than the framework table's own high-quality value (50 ms); with the framework's minimum thresholds, gaming behaves like web browsing. And context profiles break comparability: scoring developing countries on an easier profile lifts them in global rankings.

We therefore recommend keeping one common composite for comparison and making the context-specific view a **use-case profile**: for each use case, the share of tests (or the percentile) at which its minimum and high-quality thresholds are met.